# 🫁 PneumoScan AI — Google Colab
## Entraînement EfficientNetB0 + Grad-CAM

**Avant de démarrer :** Active le GPU via `Exécution > Modifier le type d'exécution > T4 GPU`

---

## ⚙️ Étape 1 — Installation des dépendances

In [ ]:
!pip install -q grad-cam kaggle tensorflow==2.13.0
print('✅ Dépendances installées')

## 📥 Étape 2 — Téléchargement du Dataset Kaggle

In [ ]:
# Place ton fichier kaggle.json dans /root/.kaggle/
# (téléchargeable depuis ton profil Kaggle > API)

import os
os.makedirs('/root/.kaggle', exist_ok=True)

# Méthode 1 : Upload manuel depuis Colab
from google.colab import files
print('⬆️  Upload ton fichier kaggle.json...')
uploaded = files.upload()

import shutil
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle configuré')

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip -d data/
print('✅ Dataset téléchargé et extrait')

# Vérification
import os
for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        path = f'data/chest_xray/{split}/{cls}'
        if os.path.exists(path):
            print(f'  {split}/{cls}: {len(os.listdir(path))} images')

## 🔍 Étape 3 — Exploration des données (EDA)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import random

# Afficher des exemples des 2 classes
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Exemples de radiographies thoraciques', fontsize=14, fontweight='bold')

for row, cls in enumerate(['NORMAL', 'PNEUMONIA']):
    folder = f'data/chest_xray/train/{cls}'
    images = random.sample(os.listdir(folder), 4)
    for col, img_name in enumerate(images):
        img = mpimg.imread(os.path.join(folder, img_name))
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].set_title(cls, color='green' if cls=='NORMAL' else 'red')
        axes[row, col].axis('off')

plt.tight_layout()
plt.show()
print('✅ EDA terminée')

## 🏗️ Étape 4 — Construction du modèle EfficientNetB0

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Config
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
CLASSES    = ['NORMAL', 'PNEUMONIA']

# Générateurs
train_gen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.10,
    width_shift_range=0.10,
    height_shift_range=0.10
)
val_gen  = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory('data/chest_xray/train', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=True)
val_data   = val_gen.flow_from_directory('data/chest_xray/val',   target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)
test_data  = test_gen.flow_from_directory('data/chest_xray/test', target_size=IMG_SIZE, batch_size=1,          class_mode='binary', shuffle=False)

# Poids de classe
n_normal    = len(os.listdir('data/chest_xray/train/NORMAL'))
n_pneumonia = len(os.listdir('data/chest_xray/train/PNEUMONIA'))
total       = n_normal + n_pneumonia
class_weight = {0: total/(2*n_normal), 1: total/(2*n_pneumonia)}
print(f'⚖️ Poids : {class_weight}')

# Modèle
base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
base.trainable = False

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model = Model(inputs=base.input, outputs=output)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)
print(f'✅ Modèle construit — {model.count_params():,} paramètres')

## 🚀 Étape 5 — Entraînement Phase 1 (base gelée)

In [ ]:
callbacks = [
    ModelCheckpoint('pneumoscan_best.h5', monitor='val_auc', mode='max', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_auc', mode='max', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

history1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)
print('✅ Phase 1 terminée')

## 🔓 Étape 6 — Fine-tuning Phase 2

In [ ]:
model = tf.keras.models.load_model('pneumoscan_best.h5')

# Libérer les 30 dernières couches
for layer in model.layers:
    if hasattr(layer, 'layers'):
        for sub in layer.layers[-30:]:
            sub.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)
print('✅ Phase 2 terminée — Fine-tuning complet')

## 📊 Étape 7 — Évaluation + Métriques médicales

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import seaborn as sns

model = tf.keras.models.load_model('pneumoscan_best.h5')
y_pred_proba = model.predict(test_data).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)
y_true = test_data.classes

print('📋 Rapport de classification :')
print(classification_report(y_true, y_pred, target_names=CLASSES))
print(f'🎯 AUC-ROC : {roc_auc_score(y_true, y_pred_proba):.4f}')

# Matrice de confusion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title('Matrice de Confusion', fontweight='bold')
axes[0].set_ylabel('Vraie étiquette')
axes[0].set_xlabel('Prédiction')

# Courbe ROC
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
auc = roc_auc_score(y_true, y_pred_proba)
axes[1].plot(fpr, tpr, color='#00e5ff', lw=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set_xlabel('Taux de Faux Positifs')
axes[1].set_ylabel('Recall (Sensibilité)')
axes[1].set_title('Courbe ROC', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 🌡️ Étape 8 — Grad-CAM sur des exemples du test set

In [ ]:
import cv2

def get_gradcam(model, img_array, layer_name='top_conv'):
    grad_model = tf.keras.Model(
        inputs=model.input,
        outputs=[model.get_layer(layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def show_gradcam(img_path, model):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224,224))
    arr = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    arr_exp = np.expand_dims(arr, 0)
    
    pred = float(model.predict(arr_exp, verbose=0)[0][0])
    label = 'PNEUMONIA' if pred >= 0.5 else 'NORMAL'
    conf  = pred*100 if pred >= 0.5 else (1-pred)*100
    
    heatmap = get_gradcam(model, arr_exp)
    
    img_cv = cv2.imread(img_path)
    img_cv = cv2.resize(img_cv, (224,224))
    img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    
    hm_resized = cv2.resize(heatmap, (224,224))
    hm_uint8   = np.uint8(255 * hm_resized)
    hm_colored = cv2.applyColorMap(hm_uint8, cv2.COLORMAP_JET)
    hm_colored = cv2.cvtColor(hm_colored, cv2.COLOR_BGR2RGB)
    overlay    = (0.4*hm_colored + 0.6*img_rgb).astype(np.uint8)
    
    fig, axes = plt.subplots(1,3, figsize=(13,4))
    color = '#ef4444' if label=='PNEUMONIA' else '#10b981'
    fig.suptitle(f'{label} — Confiance : {conf:.1f}%', fontsize=13, fontweight='bold', color=color)
    
    for ax, img_data, title in zip(axes, [img_rgb, hm_resized, overlay],
                                   ['Radio originale', 'Heatmap Grad-CAM', 'Superposition']):
        ax.imshow(img_data, cmap='jet' if title=='Heatmap Grad-CAM' else None)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'gradcam_{label}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Tester sur 2 exemples
pneumonia_imgs = os.listdir('data/chest_xray/test/PNEUMONIA')
normal_imgs    = os.listdir('data/chest_xray/test/NORMAL')

show_gradcam(f'data/chest_xray/test/PNEUMONIA/{pneumonia_imgs[0]}', model)
show_gradcam(f'data/chest_xray/test/NORMAL/{normal_imgs[0]}', model)

## 💾 Étape 9 — Télécharger le modèle
Télécharge `pneumoscan_best.h5` pour l'utiliser avec Streamlit.

In [ ]:
from google.colab import files
files.download('pneumoscan_best.h5')
files.download('evaluation_results.png')
print('✅ Fichiers téléchargés !')